# Wildlife Re-Identification | Public MAP@5 0.84435

This notebook reproduces the species-specific portfolio used for
`submssion_species_specific.csv`. It ranks five identities for each of the 2,791
queries, with `new_individual` available for animals outside the gallery.
The input is the supplied global descriptors, SIFT keypoints and descriptors,
species and viewpoint metadata. Photo IDs are used to join rows and verify
outputs.

## Model

Gallery retrieval proposes 24 identities per query. It combines cosine and
PCA similarities, shrinkage LDA, within-class covariance normalization,
KISSME, cross-view CCA, balanced ridge prototypes, Fisher/VLAD texture,
adaptive score normalization and graph propagation. Geometry uses RootSIFT
matches, a reciprocal ratio test and affine verification. MAGSAC affine and
homography checks add evidence for the high-ranked gallery photos and
query-to-query links.

LightGBM LambdaRank and histogram gradient boosting score candidate
identities. A side-aware branch adds same-side and opposite-side gallery
evidence, with logistic regression and extremely randomized trees for
novelty. A separate joint-ranking branch averages CatBoost QuerySoftMax and
XGBoost NDCG posteriors that include `new_individual`. Calibration uses
out-of-fold margins, with repeated held-out query IDs purged from the ranker
fit producing those margins.

| Species | SIFT-verified graph branch | Side-aware branch | Joint CatBoost/XGBoost |
|---|---:|---:|---:|
| Hyena | 50% | 40% | 10% |
| Leopard | 15% | 70% | 15% |


## Validation and error analysis

The following scores describe the submitted portfolio on local holdouts. The blend was selected on shared and encounter validation before scoring the stress audits.

| Holdout | Overall | Known | Unseen | Hyena | Leopard | Rare known |
|---|---:|---:|---:|---:|---:|---:|
| Shared | 0.89384 | 0.89921 | 0.87587 | 0.89040 | 0.89527 | 0.45806 |
| Encounter cluster | 0.89768 | 0.86397 | 0.94480 | 0.91772 | 0.88972 | 0.45359 |
| Ordinary | 0.89605 | 0.87227 | 0.93802 | 0.90974 | 0.88916 | 0.36984 |
| Withheld viewpoint | 0.80631 | 0.76502 | 0.97145 | 0.72525 | 0.83981 | 0.35080 |
| Encounter and camera-side stress | 0.77792 | 0.68033 | 0.92647 | 0.76768 | 0.78431 | 0.27997 |

Rare known means the identity has one or two photos in the gallery. Cross-view queries and rare gallery identities remain difficult. The most demanding encounter and camera-side audit has a known MAP@5 of 0.68033 and an unseen MAP@5 of 0.92647. The test export puts the unseen label first for 479 hyenas and 611 leopards.

## Run

Attach the competition files as a Kaggle input, or set `DATA_ROOT` to the local dataset directory. Run all cells to generate the CSV. The first run builds feature and model-margin caches from the supplied data. Later runs can reuse those caches. Use the printed SHA-256 to verify the exact scored output. Library versions can affect numerical reproducibility.


In [ ]:
import os
from pathlib import Path

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"

ON_KAGGLE = Path("/kaggle/input").is_dir()
DATA_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path.cwd()
if not ON_KAGGLE and (DATA_ROOT / "projectwild3" / "reid_student_data_main").is_dir():
    DATA_ROOT = DATA_ROOT / "projectwild3"
OUTPUT = Path("/kaggle/working") if ON_KAGGLE else Path("submission_outputs")
CACHE = OUTPUT / "portfolio_cache"
WORKERS = 5

import sys
if (DATA_ROOT / ".research_packages").is_dir():
    sys.path.insert(0, str((DATA_ROOT / ".research_packages").resolve()))

import csv
import hashlib
from collections import OrderedDict, defaultdict
from concurrent.futures import ThreadPoolExecutor
from contextlib import ExitStack
from dataclasses import dataclass, fields
from threading import Lock

import cv2
import numpy as np
from scipy import sparse
from scipy.linalg import cho_factor, cho_solve, eigh
from scipy.special import expit, logit, softmax
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.mixture import GaussianMixture
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, normalize
from catboost import CatBoostRanker
from lightgbm import LGBMRanker
from xgboost import XGBRanker

NEW = "new_individual"
SPECIES = ("hyena", "leopard")
ANGLES = {"front": 0, "frontright": 1, "right": 2, "backright": 3,
          "back": 4, "backleft": 5, "left": 6, "frontleft": 7}
SHORTLIST, SIFT_IDENTITIES, SIFT_REFS, WIDTH = 14, 9, 2, 24


## Data, retrieval features and SIFT geometry


In [ ]:
def unit(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)


def read_table(path):
    with path.open(newline="", encoding="utf-8-sig") as source:
        return list(csv.DictReader(source))


def locate_data(root, need_sift=True):
    root = Path(root)
    required = ("train.csv", "test.csv", "miewid_ids.csv", "miewid_features.npy",
                "sample_submission.csv")
    tables = [p for p in [root, root / "reid_student_data_main"]
              if all((p / name).is_file() for name in required)]
    if not tables:
        tables = [p.parent for p in sorted(root.rglob("miewid_ids.csv"))
                  if all((p.parent / name).is_file() for name in required)]
    if len(tables) != 1:
        raise ValueError(f"Expected one competition table directory under {root}; found {tables}")
    table = tables[0]
    archives = []
    filenames = ("sift_features_train.npz", "sift_features_test.npz") if need_sift else ()
    for filename in filenames:
        candidates = [table / filename, table.parent / filename, root / filename]
        found = next((p for p in candidates if p.exists()), None)
        if found is None:
            matches = sorted(root.rglob(filename))
            if len(matches) != 1:
                raise ValueError(f"Expected one supplied {filename}; found {matches}")
            found = matches[0]
        archives.append(found)
    return table, archives


class Data:
    def __init__(self, root, need_sift=True):
        table, self.archives = locate_data(root, need_sift=need_sift)
        self.train = read_table(table / "train.csv")
        self.test = read_table(table / "test.csv")
        self.sample_ids = [row["id"] for row in read_table(table / "sample_submission.csv")]
        meta = self.train + self.test
        feature_ids = [row["id"] for row in read_table(table / "miewid_ids.csv")]
        mapping = {identifier: i for i, identifier in enumerate(feature_ids)}
        ids = [row["id"] for row in meta]
        raw = np.load(table / "miewid_features.npy", allow_pickle=False)
        if (len(mapping) != len(feature_ids) or len(set(ids)) != len(ids)
                or set(feature_ids) != set(ids)
                or raw.shape != (len(meta), 2152)
                or not np.isfinite(raw).all()):
            raise ValueError("Unexpected global descriptor shape, IDs, or values")
        if set(row["species"] for row in meta) != set(SPECIES) or any(
                not row["identity"] or row["identity"] == NEW for row in self.train):
            raise ValueError("Unexpected species or empty training identities")
        self.label_species = {row["identity"]: row["species"] for row in self.train}
        if any(self.label_species[row["identity"]] != row["species"] for row in self.train):
            raise ValueError("A training identity appears in both species")
        if (len(set(self.sample_ids)) != len(self.sample_ids)
                or set(self.sample_ids) != {row["id"] for row in self.test}):
            raise ValueError("Sample IDs do not match test IDs")
        positions = np.array([mapping[identifier] for identifier in ids])
        self.x = unit(np.ascontiguousarray(raw[positions], dtype=np.float32))
        self.id = np.asarray(ids)
        self.species = np.asarray([row["species"] for row in meta])
        self.view = np.asarray([row["viewpoint"] for row in meta])
        self.angle = np.asarray([ANGLES.get(v, -1) for v in self.view])
        self.label = np.asarray([row.get("identity", "") for row in meta])
        self.train_rows = np.arange(len(self.train))
        self.test_rows = np.arange(len(self.train), len(meta))
        self.duplicates = np.asarray([
            hashlib.blake2b(v, digest_size=12).hexdigest() for v in raw[positions[:len(self.train)]]
        ])


def split(data, rows, seed, novelty=0.17, known=0.20, cross_view=False):
    """Withhold complete identities for novelty and image groups for known matches."""
    rng = np.random.default_rng(seed)
    gallery, queries = [], []
    for species in SPECIES:
        species_rows = rows[data.species[rows] == species]
        labels = data.label[species_rows]
        identities, counts = np.unique(labels, return_counts=True)
        held = set()
        buckets = np.digitize(counts, [2, 4, 8, 16, 32])
        for bucket in np.unique(buckets):
            group = identities[buckets == bucket]
            n = min(len(group) - 1, round(novelty * len(group)))
            held.update(rng.permutation(group)[:n])
        by_identity = {
            identity: species_rows[labels == identity] for identity in identities
        }
        for identity, members in by_identity.items():
            if identity in held:
                queries.extend(members)
                continue
            if len(members) < 2 or (len(members) < 5 and rng.random() > 0.65):
                gallery.extend(members)
                continue
            chosen = np.array([], dtype=int)
            if cross_view:
                sides = data.view[members]
                side_choices = [side for side in ("left", "right")
                                if np.any(sides == side) and np.any(sides != side)
                                and (sides == side).mean() <= 0.70]
                if side_choices and rng.random() < 0.70:
                    chosen = members[sides == rng.choice(side_choices)]
            if len(chosen) == 0:
                n = min(len(members) - 1, max(1, round(known * len(members))))
                chosen = rng.permutation(members)[:n]
            # Keep exact duplicate descriptors on the same side of the split.
            mask = np.isin(data.duplicates[members], data.duplicates[chosen])
            if mask.all():
                mask[:] = False
            queries.extend(members[mask])
            gallery.extend(members[~mask])
    return np.asarray(sorted(gallery)), np.asarray(sorted(queries))


class SiftStore:
    def __init__(self, data, workers):
        self.ids = data.id
        self.n_train = len(data.train)
        self.archives = [p if p.is_dir() else np.load(p, allow_pickle=False)
                         for p in data.archives]
        self.images = OrderedDict()
        self.pair_cache = {}
        self.lock = Lock()
        self.workers = workers
        cv2.setNumThreads(1)

    def close(self):
        for archive in self.archives:
            if not isinstance(archive, Path):
                archive.close()

    def load(self, row):
        with self.lock:
            if row in self.images:
                self.images.move_to_end(row)
                return self.images[row]
        archive = self.archives[int(row >= self.n_train)]
        keys = [f"{self.ids[row]}_{suffix}" for suffix in ("xy", "desc")]
        if isinstance(archive, Path):
            xy, desc = [np.load(archive / f"{key}.npy", allow_pickle=False) for key in keys]
        else:
            xy, desc = [archive[key] for key in keys]
        if xy.shape != (len(desc), 2) or desc.ndim != 2 or desc.shape[1] != 128:
            raise ValueError(f"Unexpected SIFT arrays for {self.ids[row]}")
        desc = desc.astype(np.float32)
        norm = desc.sum(axis=1)
        valid = norm > 0
        desc = desc[valid] / norm[valid, None]
        # Hellinger distance on the provided descriptors, with no image processing.
        record = (xy[valid].astype(np.float32), np.sqrt(desc, out=desc))
        with self.lock:
            self.images[row] = record
            if len(self.images) > 1400:
                self.images.popitem(last=False)
        return record

    def compare(self, pair):
        xy1, desc1 = self.load(pair[0])
        xy2, desc2 = self.load(pair[1])
        out = np.zeros(6, dtype=np.float32)
        if min(len(desc1), len(desc2)) < 4:
            return out
        out[5] = 1
        # RootSIFT has unit L2 norm, so 2(1 - dot product) is its squared distance.
        similarity = desc1 @ desc2.T
        rows = np.arange(len(desc1))
        nearest = similarity.argmax(axis=1)
        reciprocal = similarity.argmax(axis=0)[nearest] == rows
        d1 = np.maximum(1 - similarity[rows, nearest], 0)
        similarity[rows, nearest] = -np.inf
        d2 = np.maximum(1 - similarity.max(axis=1), 0)
        keep = reciprocal & (d2 > 1e-7) & (d1 < 0.84 ** 2 * d2)
        if keep.sum() < 4:
            return out
        a, b = xy1[keep], xy2[nearest[keep]]
        ratios = np.sqrt(d1[keep] / d2[keep])
        out[0] = np.log1p(len(a))
        scale = max(float(np.ptp(b, axis=0).max()), 128)
        limit = float(np.clip(0.014 * scale, 3.0, 6.0))
        for fit in (cv2.estimateAffinePartial2D, cv2.estimateAffine2D):
            transform, mask = fit(a, b, method=cv2.RANSAC, ransacReprojThreshold=limit,
                                  maxIters=700, confidence=0.995, refineIters=5)
            if transform is None or not np.isfinite(transform).all():
                continue
            stretch = np.linalg.svd(transform[:, :2], compute_uv=False)
            if stretch[1] < 0.18 or stretch[0] > 5 or stretch[0] / stretch[1] > 3.0:
                continue
            inlier = mask.ravel().astype(bool)
            n = int(inlier.sum())
            if n < 4 or np.log1p(n) <= out[1] + 1e-6:
                continue
            extent_a = np.linalg.norm(np.ptp(a[inlier], axis=0))
            extent_b = np.linalg.norm(np.ptp(b[inlier], axis=0))
            out[1:5] = (
                np.log1p(n), n / len(a), 1 - float(ratios[inlier].mean()),
                min(extent_a, extent_b) / 512,
            )
        return out

    def features(self, queries, refs):
        q, candidates, count = refs.shape
        result = np.zeros((q, candidates, count, 6), dtype=np.float32)
        requests = {}
        for i, j, k in np.argwhere(refs >= 0):
            key = tuple(sorted((int(queries[i]), int(refs[i, j, k]))))
            requests.setdefault(key, []).append((i, j, k))
        missing = sorted(requests.keys() - self.pair_cache.keys())
        if missing:
            with ThreadPoolExecutor(max_workers=self.workers) as pool:
                self.pair_cache.update(zip(missing, pool.map(self.compare, missing)))
        for key, positions in requests.items():
            for position in positions:
                result[position] = self.pair_cache[key]
        best = result[..., 1].argmax(axis=2)
        return np.concatenate([
            np.take_along_axis(result, best[:, :, None, None], axis=2).squeeze(2),
            result[..., 1].mean(axis=2, keepdims=True),
            result[..., 5].max(axis=2, keepdims=True),
        ], axis=2)


def strongest(matrix, labels, n_labels):
    # Gallery columns are ordered by label, so reduceat computes all class maxima.
    starts = np.r_[0, np.flatnonzero(np.diff(labels)) + 1]
    if len(starts) != n_labels:
        raise ValueError("Noncontiguous or missing gallery identity")
    return np.maximum.reduceat(matrix, starts, axis=1)


def label_groups(labels):
    return np.split(np.arange(len(labels)), np.flatnonzero(np.diff(labels)) + 1)


def view_affinity(data, rows, cols):
    a, b = data.angle[rows, None], data.angle[cols][None, :]
    gap = np.minimum(abs(a - b), 8 - abs(a - b))
    valid = (a >= 0) & (b >= 0)
    close = np.where(valid, 1 - gap * 0.25,
                     (data.view[rows, None] == data.view[cols][None, :])
                     & (data.view[rows, None] != ""))
    return close.astype(np.float32)


def metric_embeddings(x, labels):
    """Regularized within-class whitening and a discriminative linear projection."""
    n = len(labels)
    size = min(144, n - 1, x.shape[1])
    pca = PCA(n_components=size, svd_solver="randomized", random_state=391)
    pca.fit(x[:n])
    projected = pca.transform(x)
    groups = label_groups(labels)
    centers = np.stack([projected[idx].mean(axis=0) for idx in groups])
    residual = projected[:n] - centers[labels]
    within = residual.T @ residual / n
    centered = centers - centers.mean(axis=0)
    between = centered.T @ centered / len(groups)
    within.flat[::size + 1] += 0.9 * np.trace(within) / size + 1e-5
    values, basis = eigh(between, within, check_finite=False)
    weights = np.maximum(values, 0) ** 0.35
    metric = (projected @ basis) * weights
    return unit(metric), unit(projected / np.maximum(pca.singular_values_, 1e-5) ** 0.35)


def top(matrix, n):
    n = min(n, matrix.shape[1])
    part = np.argpartition(matrix, -n, axis=1)[:, -n:]
    values = np.take_along_axis(matrix, part, axis=1)
    order = np.argsort(-values, axis=1, kind="stable")
    return np.take_along_axis(part, order, axis=1)


def neighborhood(similarity, k=18):
    cols, vals = [], []
    for offset in range(0, len(similarity), 128):
        scores = similarity[offset:offset + 128].copy()
        rows = np.arange(len(scores))
        scores[rows, rows + offset] = -np.inf
        chosen = top(scores, k)
        cols.append(chosen)
        vals.append(np.take_along_axis(scores, chosen, axis=1))
    cols = np.vstack(cols)
    vals = np.vstack(vals)
    weight = np.exp(np.clip((vals - 0.75) / 0.10, -15, 4)).astype(np.float32)
    directed = sparse.csr_matrix((weight.ravel(), (np.repeat(np.arange(len(cols)), cols.shape[1]),
                                                  cols.ravel())), shape=similarity.shape)
    symmetric = 0.65 * directed.minimum(directed.T) + 0.35 * directed.maximum(directed.T)
    return symmetric.tocsr(), cols, vals


def sparse_rowsum(matrix):
    total = np.asarray(matrix.sum(axis=1)).ravel()
    return matrix.multiply(1 / np.maximum(total, 1e-12)[:, None]).tocsr()


def pooled_features(scores, gallery_class, candidates, side, refs, priority):
    out = np.empty((*candidates.shape, 5), dtype=np.float32)
    groups = label_groups(gallery_class)
    for label, gallery_columns in enumerate(groups):
        qi, ci = np.where(candidates == label)
        if len(qi) == 0:
            continue
        sub = np.ix_(qi, gallery_columns)
        scores_for_label = scores[sub]
        best = top(scores_for_label, 3)
        values = np.take_along_axis(scores_for_label, best, axis=1)
        first = best[:, 0]
        out[qi, ci, 0] = values[:, 0]
        out[qi, ci, 1] = values.mean(axis=1)
        out[qi, ci, 2] = (values[:, 0] - values[:, -1])
        out[qi, ci, 3] = side[qi, gallery_columns[first]]
        out[qi, ci, 4] = np.log1p(len(gallery_columns))
        active = ci < SIFT_IDENTITIES
        picked = top(priority[sub][active], SIFT_REFS)
        refs[qi[active], ci[active], :picked.shape[1]] = gallery_columns[picked]
    # The tail of the shortlist gets one geometric check. For well-separated
    # global leaders, verify the nearest few alternatives rather than every tail.
    refs[:, 4:, 1] = -1
    secure = (out[:, 0, 0] > 0.67) & (
        out[:, 0, 0] - np.max(out[:, 1:, 0], axis=1) > 0.18)
    refs[secure, 4:] = -1
    return out


@dataclass
class Episode:
    rows: np.ndarray
    labels: np.ndarray
    species: np.ndarray
    base_labels: np.ndarray
    base_scores: np.ndarray
    candidates: np.ndarray
    global_features: np.ndarray
    query_features: np.ndarray
    sift_features: np.ndarray | None
    gallery_count: np.ndarray
    gallery_has_view: np.ndarray

    @property
    def novel(self):
        return self.labels == NEW


def episode_for_species(data, gallery, query, store=None, transductive=True):
    classes, gal_label = np.unique(data.label[gallery], return_inverse=True)
    order = np.argsort(gal_label, kind="stable")
    gallery, gal_label = gallery[order], gal_label[order]
    n, nq = len(gallery), len(query)
    count = len(classes)
    rows = np.r_[gallery, query]
    x = data.x[rows]
    raw = x[n:] @ x[:n].T
    base_values = strongest(raw, gal_label, count)
    base = top(base_values, SHORTLIST)
    centered = unit(x - x[:n].mean(axis=0))
    by_view = centered.copy()
    for view in np.unique(data.view[gallery]):
        use = data.view[gallery] == view
        if view and use.sum() >= 15:
            center = centered[:n][use].mean(axis=0)
            by_view[data.view[rows] == view] -= 0.8 * use.sum() / (use.sum() + 75) * center
    by_view = unit(by_view)
    metric, pca = metric_embeddings(centered, gal_label)
    broad = 0.50 * centered + 0.25 * by_view + 0.25 * x
    # Concatenated weighted embeddings produce the average of component similarities.
    weights = np.sqrt(np.array([0.54, 0.20, 0.16, 0.10], dtype=np.float32))
    retrieval = np.concatenate([weight * vectors for weight, vectors
                                in zip(weights, (centered, by_view, x, pca))], axis=1)
    similarity = retrieval @ retrieval.T
    side = view_affinity(data, query, gallery)
    direct = similarity[n:, :n].copy()
    supervised = metric[n:] @ metric[:n].T
    graph, neighbor_cols, neighbor_vals = neighborhood(similarity)
    del similarity, retrieval
    membership = sparse.csr_matrix((np.ones(n, dtype=np.float32),
                                   (np.arange(n), gal_label)), shape=(n, count))
    if transductive:
        propagation = sparse_rowsum(graph[n:])
        q_to_g = propagation[:, :n]
        q_to_q = propagation[:, n:]
    else:
        q_to_g = sparse_rowsum(graph[n:, :n])
        q_to_q = sparse.csr_matrix((nq, nq), dtype=np.float32)
    direct_flow = (q_to_g @ membership).toarray()
    flow = direct_flow.copy()
    for _ in range(5):
        flow = direct_flow + 0.83 * (q_to_q @ flow)
    affinity = sparse_rowsum(graph)
    profiles = affinity[n:] @ affinity[:n].T
    shared = profiles.toarray()
    del profiles
    shared /= np.sqrt(
        np.maximum(np.asarray(affinity[n:].power(2).sum(axis=1)), 1e-9)
        * np.maximum(np.asarray(affinity[:n].power(2).sum(axis=1)).T, 1e-9)
    )
    pair = 0.86 * direct + 0.14 * supervised + 0.02 * side
    pair_class = strongest(pair, gal_label, count)
    shared_class = strongest(shared, gal_label, count)
    flow_cond = flow / np.maximum(flow.sum(axis=1, keepdims=True), 1e-8)
    combined = pair_class + 0.18 * np.sqrt(flow_cond) + 0.07 * shared_class
    # Admit diverse candidates from the learned metric, direct match and graph.
    shortlist = np.empty((nq, min(SHORTLIST, count)), dtype=np.int32)
    other = [top(pair_class, 5), top(flow_cond, 4),
             top(strongest(supervised, gal_label, count), 4), base[:, :4]]
    lead = top(combined, shortlist.shape[1])
    for i in range(nq):
        stream = [*lead[i, :8], *(label for ranks in other for label in ranks[i]), *lead[i, 8:]]
        shortlist[i] = list(dict.fromkeys(stream))[:shortlist.shape[1]]
    row_ids = np.arange(nq)[:, None]
    refs = np.full((*shortlist.shape, SIFT_REFS), -1, dtype=np.int32)
    pooled = pooled_features(pair, gal_label, shortlist, side, refs,
                             pair + 0.06 * side + 0.035 * shared)
    center = np.stack([broad[idx].mean(axis=0) for idx in label_groups(gal_label)])
    prototype = unit(broad[n:]) @ unit(center).T
    global_features = np.stack([
        base_values[row_ids, shortlist],
        pair_class[row_ids, shortlist],
        strongest(direct, gal_label, count)[row_ids, shortlist],
        strongest(supervised, gal_label, count)[row_ids, shortlist],
        strongest(pca[n:] @ pca[:n].T, gal_label, count)[row_ids, shortlist],
        shared_class[row_ids, shortlist],
        np.sqrt(flow_cond[row_ids, shortlist]),
        flow[row_ids, shortlist],
        prototype[row_ids, shortlist],
        combined[row_ids, shortlist],
    ], axis=2)
    global_features = np.concatenate([global_features, pooled], axis=2)
    global_features = np.concatenate([
        global_features, global_features[:, :, :10] - global_features[:, :, :10].max(
            axis=1, keepdims=True),
    ], axis=2)
    query_features = np.column_stack([
        neighbor_vals[n:, 0], neighbor_vals[n:, -1],
        (neighbor_cols[n:] >= n).mean(axis=1),
        flow.sum(axis=1), direct_flow.sum(axis=1),
        (data.view[query] == "").astype(float), (data.species[query] == "hyena").astype(float),
        pair_class.max(axis=1), base_values.max(axis=1),
        supervised.max(axis=1), shared_class.max(axis=1),
    ]).astype(np.float32)
    counts = np.bincount(gal_label)
    lookup = {label: counts[i] for i, label in enumerate(classes)}
    gallery_count = np.array([lookup.get(label, 0) for label in data.label[query]])
    gallery_views = set(zip(data.label[gallery], data.view[gallery]))
    gallery_has_view = np.array([(label, view) in gallery_views and bool(view)
                                 for label, view in zip(data.label[query], data.view[query])])
    truth = np.where(gallery_count > 0, data.label[query], NEW)
    if store is not None:
        refs = np.where(refs >= 0, gallery[refs], -1)
        sift_features = store.features(query, refs)
    else:
        sift_features = None
    return Episode(query, truth, data.species[query], classes[base],
                   base_values[row_ids, base], classes[shortlist],
                   global_features, query_features, sift_features, gallery_count,
                   gallery_has_view)


def stack(pieces):
    return Episode(**{
        name: (None if getattr(pieces[0], name) is None
               else np.concatenate([getattr(ep, name) for ep in pieces], axis=0))
        for name in Episode.__dataclass_fields__
    })


def write_submission(data, ep, pred, path):
    by_row = dict(zip(data.id[ep.rows], pred))
    output_species = {row["id"]: row["species"] for row in data.test}
    if len(by_row) != len(ep.rows) or set(by_row) != set(data.sample_ids):
        raise ValueError("Missing or repeated output IDs")
    for image, labels in by_row.items():
        if not 1 <= len(labels) <= 5 or len(set(labels)) != len(labels):
            raise ValueError(f"Invalid number of distinct guesses at {image}")
        if any(label != NEW and data.label_species.get(label) != output_species[image]
               for label in labels):
            raise ValueError(f"Out-of-species prediction at {image}")
    temp = path.with_suffix(".tmp")
    with temp.open("w", newline="", encoding="utf-8") as destination:
        writer = csv.writer(destination, lineterminator="\n")
        writer.writerow(("id", "predictions"))
        writer.writerows((image, " ".join(by_row[image])) for image in data.sample_ids)
    temp.replace(path)


In [ ]:
def calibration_splits(data, gallery):
    return [split(data, gallery, seed, cross_view=(i == 1))
            for i, seed in enumerate((119, 222, 325))]


def query_edges(data, q, centered, store):
    n = len(q)
    if n < 2:
        return np.zeros((n, 1), int), np.zeros((n, 1), np.float32)
    sim = centered @ centered.T
    np.fill_diagonal(sim, -np.inf)
    neighbors = top(sim, min(4, n-1))
    cosine = np.take_along_axis(sim, neighbors, axis=1)
    # Verify query-to-query edges using the same supplied SIFT descriptors.
    geo = store.features(q, q[neighbors][:, :, None])
    geometry = expit(4.0 * (geo[:, :, 1] - np.log(9))) * np.clip(geo[:, :, 4]*6, 0, 1)
    support = np.maximum(expit((cosine - 0.68) / 0.048), geometry)
    return neighbors, support.astype(np.float32)


def kernel_features(data, gallery, query):
    labels, group = np.unique(data.label[gallery], return_inverse=True)
    x = unit(data.x[np.r_[gallery,query]] - data.x[gallery].mean(axis=0))
    n = len(gallery)
    g, q = x[:n], x[n:]
    gg, qg = g @ g.T, q @ g.T
    response = np.eye(len(labels), dtype=np.float32)[group]
    counts = response.sum(axis=0)
    # Fit two independent kernel ridge classifiers on the actual gallery.
    def ridge(k_train, k_query, penalty):
        k_train.flat[::n+1] += penalty
        return k_query @ cho_solve(cho_factor(k_train, lower=True, check_finite=False),
                                   response, check_finite=False)
    linear = ridge(gg.copy(), qg, 0.45)
    polynomial = ridge(0.45*gg + 0.55*gg**3, 0.45*qg + 0.55*qg**3, 0.18)
    # Compare a match against each reference's strongest other-animal matches.
    gg[group[:, None] == group[None, :]] = -2
    impostor_level = np.take_along_axis(gg, top(gg, 5), axis=1).mean(axis=1)
    corrected = qg - 0.42 * impostor_level[None, :]
    order = np.argsort(group, kind="stable")
    ordered = group[order]
    direct = strongest(qg[:, order], ordered, len(labels))
    local = strongest(corrected[:, order], ordered, len(labels))
    centroids = q @ unit((response.T @ g) / counts[:,None]).T
    # Pool several reference photos with a correction for catalogue size.
    density = np.log(np.maximum(np.exp(12*qg) @ response, 1e-15)) / 12
    density -= 0.5*np.log(counts)[None, :]/12
    raw = np.stack([direct, local, linear, polynomial, centroids, density],axis=2)
    # Include the top alternatives from both the supervised and retrieval experts.
    leaders = [top(raw[:,:,i], 12) for i in (3, 1, 0, 2, 5, 4)]
    width = min(24, len(labels))
    candidates = np.empty((len(q),width), dtype=int)
    for i in range(len(q)):
        sequence = np.stack([item[i] for item in leaders], axis=1).ravel()
        unique = list(dict.fromkeys(sequence))
        if len(unique) < width:
            unique = list(dict.fromkeys([*unique, *np.argsort(-polynomial[i])[:width]]))
        candidates[i] = unique[:width]
    features = raw[np.arange(len(q))[:,None], candidates]
    features = np.concatenate([features,
                               np.log1p(counts[candidates])[:,:,None]],axis=2)
    qs = []
    for i in range(raw.shape[2]):
        v = np.take_along_axis(raw[:,:,i], top(raw[:,:,i], 4), axis=1)
        qs.extend([v[:,0],v[:,0]-v[:,1],v[:,0]-v[:,3]])
    return labels[candidates], features, np.column_stack(qs), q


def prepare_one(data, gallery, query, store, directory, key):
    path = directory / (key + ".npz")
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            if not (np.array_equal(saved["gallery_ids"], data.id[gallery]) and
                    np.array_equal(saved["query_ids"],data.id[query])):
                raise ValueError(f"Cache split mismatch for {key}")
        return
    print("Preparing", key, len(gallery), len(query), flush=True)
    base, labels, feats, query_features, links, reliability = [], [], [], [], [], []
    offset = 0
    for sp in SPECIES:
        g = gallery[data.species[gallery]==sp]
        q = query[data.species[query]==sp]
        base.append(episode_for_species(data,g,q,store))
        c, features, signals, centered = kernel_features(data,g,q)
        neighbors, weights = query_edges(data,q,centered,store)
        labels.append(c)
        feats.append(features)
        query_features.append(signals)
        links.append(neighbors+offset)
        reliability.append(weights)
        offset += len(q)
        print(" ",key,sp,"ready",flush=True)
    ep = stack(base)
    values = {f.name: getattr(ep,f.name) for f in fields(ep)}
    temp = path.with_suffix(".partial.npz")
    np.savez_compressed(temp, **values, kernel_labels=np.concatenate(labels),
                        kernel_x=np.concatenate(feats), kernel_q=np.concatenate(query_features),
                        neighbors=np.concatenate(links), weight=np.concatenate(reliability),
                        gallery_ids=data.id[gallery], query_ids=data.id[query])
    temp.replace(path)


def prepare(data, fold_map, store, directory, names):
    directory.mkdir(parents=True,exist_ok=True)
    for name in names:
        prepare_one(data,*fold_map[name],store,directory,name)


def read_episode(directory, key):
    with np.load(directory / (key + ".npz"), allow_pickle=False) as file:
        ep = Episode(**{f.name: file[f.name] for f in fields(Episode)})
        extras = {k:file[k] for k in ("kernel_labels","kernel_x","kernel_q","neighbors","weight")}
    return ep, extras


def dense_posterior(parts, species, vocabulary):
    labels, conditional, odds = parts
    pi = {s:estimate_prior(odds[species==s]) for s in SPECIES}
    p_new = expit(odds + logit(np.array([pi[s] for s in species])))
    p = np.zeros((len(labels),len(vocabulary)+1), dtype=np.float64)
    p[np.arange(len(labels))[:,None],np.searchsorted(vocabulary,labels)] = (
        conditional*(1-p_new[:,None]))
    p[:,-1] = p_new
    return p


def ranked(prob, vocabulary):
    labels = np.r_[vocabulary,np.array([NEW])]
    return labels[top(prob,5)]


def validate_inputs(directory,key,fold_map,data):
    episode = read_episode(directory,key)
    ep = episode[0]
    gallery, query = fold_map[key]
    if set(ep.rows) != set(query) or len(ep.rows)!=len(query) or np.intersect1d(gallery,query).size:
        raise ValueError(f"Invalid split for {key}")
    if key!="test" and np.intersect1d(data.duplicates[query],data.duplicates[gallery]).size:
        raise ValueError("Cross-split duplicate")
    return episode


def possible_labels(episode):
    ep, extra = episode
    return np.union1d(ep.candidates, extra["kernel_labels"])


In [ ]:
class KeypointSource:

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.stack.close()

    def load(self, row):
        archive = self.archives[int(row >= len(self.data.train))]
        image = self.data.id[row]

        def read(suffix):
            key = f'{image}_{suffix}'
            if isinstance(archive, Path):
                return np.load(archive / (key + '.npy'), allow_pickle=False)
            return archive[key]
        xy, desc = (read('xy'), read('desc'))
        valid = desc.sum(axis=1) > 0
        return (xy[valid].astype('f'), desc[valid])

def affine_backend():
    x = np.array([[0,0],[20,0],[0,20],[20,20],[10,14],[30,21]],dtype="f")
    try:
        model,_ = cv2.estimateAffinePartial2D(x,x+1,method=cv2.USAC_MAGSAC)
        if model is not None:
            return "partial_affine_magsac"
    except cv2.error:
        pass
    return "full_affine_magsac_projected_to_similarity"


class Verification(KeypointSource):
    def __init__(self,data,workers,cache):
        self.data = data
        self.workers = workers
        self.stack = ExitStack()
        self.archives = [path if path.is_dir() else self.stack.enter_context(np.load(path,allow_pickle=False))
                         for path in data.archives]
        self.images = {}
        self.overflow = {}
        self.matches = OrderedDict()
        cv2.setNumThreads(1)
        self.backend = affine_backend()
        tag = hashlib.sha256(data.id.tobytes()).hexdigest()[:16]
        directory = cache / f"rootsift_input_{tag}"
        directory.mkdir(exist_ok=True)
        self.count = self.open_array(directory/"counts.npy",(len(data.id),),np.int16,-1)
        self.descriptors = self.open_array(directory/"descriptors.npy",(len(data.id),500,128),np.uint8)
        self.positions = self.open_array(directory/"positions.npy",(len(data.id),500,2),np.float32)

    @staticmethod
    def open_array(path,shape,dtype,fill=None):
        if path.exists():
            array = np.load(path,mmap_mode="r+",allow_pickle=False)
            if array.shape!=shape or array.dtype!=dtype:
                raise ValueError(f"Invalid descriptor cache {path}")
        else:
            array = np.lib.format.open_memmap(path,mode="w+",shape=shape,dtype=dtype)
            if fill is not None:
                array.fill(fill)
                array.flush()
        return array

    def get(self,row):
        if row in self.overflow:
            return self.overflow[row]
        count = int(self.count[row])
        assert count>=0
        return self.positions[row,:count],self.descriptors[row,:count]

    def stage_images(self,images,workers):
        self.overflow.clear()
        missing = sorted(row for row in images if self.count[row]<0)
        for offset in range(0,len(missing),128):
            batch = missing[offset:offset+128]
            written = []
            for row,(xy,desc) in zip(batch,workers.map(self.load,batch)):
                count = len(desc)
                compact = desc.astype(np.uint8,copy=False)
                if count>500 or (desc.dtype!=np.uint8 and not np.array_equal(desc,compact)):
                    # Preserve atypical arrays exactly in the current batch.
                    self.overflow[row] = xy,desc
                    continue
                self.positions[row,:count] = xy
                self.descriptors[row,:count] = compact
                written.append((row,count))
            self.positions.flush()
            self.descriptors.flush()
            for row,count in written:
                self.count[row] = count
            self.count.flush()

    def compare(self,pair):
        xy1, desc1 = self.get(pair[0])
        xy2, desc2 = self.get(pair[1])
        if min(len(desc1),len(desc2)) < 6:
            return (0,0.,0.)
        def root(matrix):
            return np.sqrt(matrix / matrix.sum(axis=1,keepdims=True,dtype=np.float32))
        sim = root(desc1) @ root(desc2).T
        nearest = sim.argmax(axis=1)
        source = np.arange(len(sim))
        mutual = sim.argmax(axis=0)[nearest] == source
        first = (1-sim[source,nearest]).clip(0)
        sim[source,nearest] = -2
        second = (1-sim.max(axis=1)).clip(0)
        keep = mutual & (second > 1e-7) & (first < .84**2*second)
        if keep.sum()<6:
            return (0,0.,0.)
        a,b = xy1[keep],xy2[nearest[keep]]
        limit = float(np.clip(.014*max(np.ptp(b,axis=0)),3,6))
        results = []
        fit = cv2.estimateAffinePartial2D if self.backend.startswith("partial") else cv2.estimateAffine2D
        transform, mask = fit(a,b,method=cv2.USAC_MAGSAC,ransacReprojThreshold=limit,
                              maxIters=700,confidence=.995)
        if transform is not None and mask is not None and np.isfinite(transform).all():
            u,s,v = np.linalg.svd(transform[:,:2])
            affine_inliers = mask.ravel().astype(bool)
            if .20<=s[1]<=s[0]<=4. and s[0]/s[1]<1.5 and np.linalg.det(u@v)>0 and affine_inliers.sum()>=6:
                rotation = (u@v)*s.mean()
                offset = np.median(b[affine_inliers]-a[affine_inliers]@rotation.T,axis=0)
                results.append(np.linalg.norm(a@rotation.T+offset-b,axis=1)<=limit)
        projective, mask = cv2.findHomography(a,b,method=cv2.USAC_MAGSAC,
                            ransacReprojThreshold=limit,maxIters=700,confidence=.995)
        if projective is not None and mask is not None and np.isfinite(projective).all():
            aa = cv2.perspectiveTransform(a[None],projective)[0]
            # Check the inverse and the local Jacobian to reject unstable warps.
            try:
                inverse = np.linalg.inv(projective)
            except np.linalg.LinAlgError:
                inverse = None
            span = np.ptp(a,axis=0).clip(1)
            center = np.median(a,axis=0)
            probe = np.vstack([center,center+[.05*span[0],0],center+[0,.05*span[1]]]).astype("f")
            transformed = cv2.perspectiveTransform(probe[None],projective)[0]
            jac = ((transformed[1:]-transformed[0])/(.05*span[:,None])).T
            if inverse is not None and np.isfinite(jac).all() and np.linalg.det(jac)>0:
                singular = np.linalg.svd(jac,compute_uv=False)
                if .20<=singular[1]<=singular[0]<=4. and singular[0]/singular[1]<2.4:
                    bb = cv2.perspectiveTransform(b[None],inverse)[0]
                    results.append(mask.ravel().astype(bool) & (np.linalg.norm(aa-b,axis=1)<=limit)
                                   & (np.linalg.norm(bb-a,axis=1)<=limit/singular.mean()))
        best = (0,0.,0.)
        for mask in results:
            count = int(mask.sum())
            if count < 6:
                continue
            spread = min(np.linalg.norm(np.ptp(x[mask],axis=0))/
                         max(np.linalg.norm(np.ptp(all_points,axis=0)),1.)
                         for x,all_points in ((a,xy1),(b,xy2)))
            best = max(best,(count,count/len(a),float(spread)))
        return best

    def features(self,query,gallery,shortlist,candidates,checkpoint):
        qg = gallery[shortlist]
        pairs = [[tuple(sorted((int(q),int(g)))) for g in photos] for q,photos in zip(query,qg)]
        signature = hashlib.sha256(query.tobytes()+qg.tobytes()+candidates.tobytes()).hexdigest()
        result = np.zeros((*candidates.shape,4),dtype="f")
        start = 0
        if checkpoint.exists():
            with np.load(checkpoint,allow_pickle=False) as stored:
                assert str(stored["signature"])==signature
                result = stored["features"]
                start = int(stored["complete"])
        print("Top-50 SIFT",len(query),"queries,",start,"previously completed",flush=True)
        labels = self.data.label[qg]
        with ThreadPoolExecutor(max_workers=self.workers) as workers:
            for offset in range(start,len(query),64):
                end = min(offset+64,len(query))
                todo = sorted({pair for block in pairs[offset:end] for pair in block}-self.matches.keys())
                self.stage_images({row for pair in todo for row in pair},workers)
                for index in range(0,len(todo),256):
                    batch = todo[index:index+256]
                    self.matches.update(zip(batch,workers.map(self.compare,batch)))
                for i in range(offset,end):
                    groups = defaultdict(list)
                    for label,pair in zip(labels[i],pairs[i]):
                        groups[label].append(self.matches[pair])
                    for j,label in enumerate(candidates[i]):
                        if label in groups:
                            values = np.asarray(groups[label])
                            result[i,j] = [values[:,0].max(),np.sort(values[:,0])[-2:].sum(),
                                           values[:,1].max(),values[:,2].max()]
                while len(self.matches)>32000:
                    self.matches.popitem(last=False)
                temp = checkpoint.with_suffix(".partial.npz")
                np.savez_compressed(temp,signature=signature,features=result,complete=end)
                temp.replace(checkpoint)
                if end%256==0 or end==len(query):
                    print("Verified",end,"/",len(query),"queries",flush=True)
        return result


def training_positions(ep):
    eligible = np.flatnonzero((ep.candidates == ep.labels[:,None]).any(axis=1))
    # Positive-known queries are the original identity-ranker training population.
    # Stratify the subset for exact geometry by species and keep all legacy inputs.
    rng = np.random.default_rng(83484)
    positions = []
    for species in SPECIES:
        keep = eligible[ep.species[eligible]==species]
        cap = {"hyena":170,"leopard":300}[species]
        positions.extend(rng.choice(keep,min(len(keep),cap),replace=False))
    return np.sort(positions)

def prepare_geometry(data, folds, key, verifier):
    path = CACHE / f"gallery_sift_{key}.npz"
    if path.exists():
        return
    ep, _ = validate_inputs(CACHE, key, folds, data)
    positions = np.arange(len(ep.rows)) if key == "test" else training_positions(ep)
    rows, candidates = ep.rows[positions], ep.candidates[positions]
    gallery = folds[key][0]
    features = np.zeros((*candidates.shape, 7), dtype="f")
    for species in SPECIES:
        take = np.flatnonzero(data.species[rows] == species)
        g = gallery[data.species[gallery] == species]
        g = g[np.argsort(data.label[g], kind="stable")]
        q = rows[take]
        shortlist = top(data.x[q] @ data.x[g].T, 50)
        checkpoint = CACHE / f"gallery_sift_{key}_{species}_geometry.npz"
        features[take, :, 3:] = verifier.features(q, g, shortlist, candidates[take], checkpoint)
    signature = hashlib.sha256(data.id[gallery].tobytes() + data.id[rows].tobytes()
                               + candidates.tobytes()).hexdigest()
    temporary = path.with_suffix(".partial.npz")
    np.savez_compressed(temporary, signature=signature, features=features)
    temporary.replace(path)


## Texture and supervised retrieval


In [ ]:
DICTIONARY_IDS = (
    'tr_00076', 'tr_00123', 'tr_00154', 'tr_00238', 'tr_00273', 'tr_00366', 'tr_00367', 'tr_00399',
    'tr_00493', 'tr_00511', 'tr_00546', 'tr_00580', 'tr_00623', 'tr_01023', 'tr_01157', 'tr_01239',
    'tr_01338', 'tr_01357', 'tr_01395', 'tr_01474', 'tr_01504', 'tr_01566', 'tr_01681', 'tr_01895',
    'tr_01965', 'tr_02004', 'tr_02129', 'tr_02281', 'tr_02316', 'tr_02319', 'tr_02391', 'tr_02508',
    'tr_02595', 'tr_02747', 'tr_02890', 'tr_02961', 'tr_03017', 'tr_03118', 'tr_03135', 'tr_03192',
    'tr_03358', 'tr_03367', 'tr_03437', 'tr_03602', 'tr_03822', 'tr_04034', 'tr_04075', 'tr_04107',
    'tr_04123', 'tr_04294', 'tr_04353', 'tr_04494', 'tr_04545', 'tr_04827', 'tr_04862', 'tr_04977',
    'tr_05038', 'tr_05053', 'tr_05114', 'tr_05129', 'tr_05144', 'tr_05146', 'tr_05225', 'tr_05314',
    'tr_05354', 'tr_05369', 'tr_05370', 'tr_05439', 'tr_05456', 'tr_05462', 'tr_05464', 'tr_05533',
    'tr_05560', 'tr_05647', 'tr_05825', 'tr_05863', 'tr_05888', 'tr_05927', 'tr_05969', 'tr_05970',
    'tr_05982', 'tr_06004', 'tr_06019', 'tr_06069', 'tr_06256', 'tr_06326', 'tr_06469', 'tr_06541',
    'tr_06667', 'tr_06859', 'tr_06937', 'tr_07004', 'tr_07026', 'tr_07123', 'tr_07136', 'tr_07147',
)

def root(desc):
    desc = desc.astype(np.float32)
    return np.sqrt(desc/desc.sum(axis=1,keepdims=True).clip(1e-12))


def encode(desc,pca,mix):
    x = pca.transform(root(desc))
    q = mix.predict_proba(x)
    diff = (x[:,None,:]-mix.means_[None])/np.sqrt(mix.covariances_[None])
    weight = q[:,:,None]
    a = (weight*diff).mean(0)/np.sqrt(mix.weights_[:,None])
    b = (weight*(diff**2-1)).mean(0)/np.sqrt(2*mix.weights_[:,None])
    hard = q.argmax(1)
    vlad = np.zeros_like(mix.means_)
    np.add.at(vlad,hard,x-mix.means_[hard])
    vlad = unit(vlad)
    def normalize(z):
        z = np.sign(z)*np.sqrt(np.abs(z))
        return z/max(np.linalg.norm(z),1e-12)
    return normalize(np.r_[a.ravel(),b.ravel()]),normalize(vlad.ravel())

def texture_codes(data, folds, keys, verifier):
    path = CACHE / "texture_codes.npz"
    index = {name: i for i, name in enumerate(data.id)}
    common = np.array([index[name] for name in DICTIONARY_IDS])
    for key in keys:
        if not np.isin(common, folds[key][0]).all():
            raise ValueError("Texture dictionary rows must belong to the gallery")
    if not path.exists():
        rng = np.random.default_rng(642)
        def load(row):
            return verifier.get(row)[1] if verifier.count[row] >= 0 else verifier.load(row)[1]
        samples = []
        for row in common:
            desc = load(row)
            if len(desc):
                samples.append(root(desc[rng.choice(len(desc), min(len(desc), 180), replace=False)]))
        local = np.concatenate(samples)
        pca = PCA(n_components=32, svd_solver="randomized", random_state=642)
        z = pca.fit_transform(local)
        mix = GaussianMixture(n_components=24, covariance_type="diag", max_iter=55,
                              reg_covar=1e-5, random_state=642).fit(z)
        fisher = np.zeros((len(data.id),1536), dtype="float16")
        vlad = np.zeros((len(data.id),768), dtype="float16")
        for row in range(len(data.id)):
            desc = load(row)
            if len(desc):
                fisher[row], vlad[row] = encode(desc, pca, mix)
        np.savez(path, ids=data.id, fisher=fisher, vlad=vlad, dictionary_ids=data.id[common])
    with np.load(path, allow_pickle=False) as cache:
        np.testing.assert_array_equal(cache["ids"], data.id)
        np.testing.assert_array_equal(cache["dictionary_ids"], DICTIONARY_IDS)
        return cache["fisher"], cache["vlad"]


In [ ]:
def fit_wccn(gallery, groups, query, reg=.06, power=.50):
    residuals = np.vstack([gallery[idx] - gallery[idx].mean(axis=0, keepdims=True)
                           for idx in groups if len(idx) >= 2])
    values, basis = np.linalg.eigh((residuals.T @ residuals) / float(len(residuals)))
    projection = (basis * ((np.maximum(0.0, values) + reg) ** -power)[None, :]) @ basis.T
    return (normalize(gallery @ projection, axis=1).astype(np.float32),
            normalize(query @ projection, axis=1).astype(np.float32))


def fit_kissme(gallery, groups, query, reg_s=.12, reg_d=.12, max_dim=256):
    d = min(max_dim, gallery.shape[1])
    g = normalize(gallery[:, -d:], axis=1).astype(np.float64)
    q = normalize(query[:, -d:], axis=1).astype(np.float64)
    residuals = []
    for idx in groups:
        count = len(idx)
        if count >= 2:
            examples = g[idx]
            factor = np.sqrt(float(count)) / float(count)
            residuals.append((examples - examples.mean(0, keepdims=True)) * np.sqrt(factor * count))
    joined = np.vstack(residuals)
    same_covariance = 2.0 * (joined.T @ joined) / float(len(joined))
    centered = g - g.mean(0, keepdims=True)
    other_covariance = 2.0 * (centered.T @ centered) / float(len(g))
    matrix = np.linalg.inv(same_covariance + reg_s * np.eye(d)) - np.linalg.inv(other_covariance + reg_d * np.eye(d))
    values, basis = np.linalg.eigh(0.5 * (matrix + matrix.T))
    positive = values > 1e-6
    projection = (basis[:, positive] * np.sqrt(values[positive])[None, :] if positive.sum() >= 16
                  else basis[:, -min(64,d):])
    return normalize(g @ projection, axis=1).astype(np.float32), normalize(q @ projection, axis=1).astype(np.float32)


def fit_cross_view_cca(gallery, groups, gallery_views, query, query_views, reg=.18, max_dim=224, n_comp=96):
    left = ("left", "frontleft", "backleft")
    right = ("right", "frontright", "backright")
    d = min(max_dim, gallery.shape[1])
    g = normalize(gallery[:, -d:], axis=1).astype(np.float64)
    q = normalize(query[:, -d:], axis=1).astype(np.float64)
    paired = []
    for idx in groups:
        li = idx[np.isin(gallery_views[idx], left)]
        ri = idx[np.isin(gallery_views[idx], right)]
        if len(li) and len(ri):
            paired.append((g[li].mean(axis=0), g[ri].mean(axis=0)))
    if len(paired) < 20:
        return gallery[:, -n_comp:].astype(np.float32), query[:, -n_comp:].astype(np.float32)
    l, r = [np.vstack(side) for side in zip(*paired)]
    mean_l, mean_r = l.mean(0, keepdims=True), r.mean(0, keepdims=True)
    l, r = l - mean_l, r - mean_r
    count = float(len(l))
    def inverse_root(samples):
        values, basis = np.linalg.eigh((samples.T @ samples) / count + reg * np.eye(d))
        return (basis * (np.maximum(values, 1e-8) ** -0.5)[None, :]) @ basis.T
    norm_l, norm_r = inverse_root(l), inverse_root(r)
    u, correlation, vt = np.linalg.svd(norm_l @ ((l.T @ r) / count) @ norm_r, full_matrices=False)
    k = min(n_comp, len(correlation))
    weight = np.sqrt(np.clip(correlation[:k], 0.0, 1.0))[None, :]
    proj_l = (norm_l @ u[:, :k]) * weight
    proj_r = (norm_r @ vt[:k, :].T) * weight
    middle = 0.5 * (proj_l + proj_r)
    mean = 0.5 * (mean_l + mean_r)
    def project(x, views):
        out = (x - mean) @ middle
        for side, center, projection in ((left, mean_l, proj_l), (right, mean_r, proj_r)):
            use = np.isin(views, side)
            if np.any(use):
                out[use] = (x[use] - center) @ projection
        return normalize(out, axis=1).astype(np.float32)
    return project(g, gallery_views), project(q, query_views)


In [ ]:
def candidate_features(data, gallery, episode):
    gallery_views = set(zip(data.label[gallery], data.view[gallery]))
    supported = np.array([[bool(view) and (label, view) in gallery_views for label in labels]
                          for view, labels in zip(data.view[episode.rows], episode.candidates)], dtype=np.float32)
    sift = episode.sift_features
    relative = sift[:, :, :2] - sift[:, :, :2].max(axis=1, keepdims=True)
    return np.concatenate([episode.global_features, sift, relative, supported[:, :, None]], axis=2)


def light_model(iterations):
    return LGBMRanker(n_estimators=iterations, learning_rate=.055, num_leaves=15,
                      min_child_samples=64, reg_lambda=9.0, colsample_bytree=.85,
                      verbosity=-1, n_jobs=2, random_state=885,
                      lambdarank_truncation_level=6, label_gain=[0,1], objective="lambdarank")


def retrieval_fold(data, folds, cache, key):
    cached = validate_inputs(cache, key, folds, data)
    ep, extra = cached
    x = candidate_features(data, folds[key][0], ep)
    matches = ep.candidates[:, :, None] == extra["kernel_labels"][:, None, :]
    ri = np.arange(len(ep.rows))[:, None]
    k = extra["kernel_x"][ri, matches.argmax(axis=2)]
    k = np.where(matches.any(axis=2)[:, :, None], k, np.nan)
    relative = k - np.max(extra["kernel_x"], axis=1, keepdims=True)
    x = np.concatenate([x, k, relative], axis=2)
    # Per-query evidence helps the rankers interpret geometric and global scores.
    geometry = ep.sift_features[:, :, [1, 2, 4]].max(axis=1)
    whole = np.column_stack([ep.query_features, geometry, extra["kernel_q"]])
    broadcast = np.broadcast_to(whole[:, None], (*x.shape[:2], whole.shape[1]))
    candidate_x = np.concatenate([x, broadcast], axis=2).astype(np.float32)
    return cached, candidate_x, whole


def histogram():
    return HistGradientBoostingClassifier(max_iter=115, learning_rate=0.07,
                                          max_leaf_nodes=15, min_samples_leaf=50,
                                          l2_regularization=7, early_stopping=False,
                                          random_state=885)


def light_predict(model, x):
    return model.booster_.predict(x, raw_score=True, num_threads=2)

def snorm(qg, gg, gallery_labels, k=30):
    # Gallery cohorts exclude the reference identity. Query cohorts exclude the
    # provisional winning identity, using only observable gallery labels.
    gg[gallery_labels[:,None]==gallery_labels] = -np.inf
    g_cohort = np.take_along_axis(gg,top(gg,k),axis=1)
    leaders = gallery_labels[qg.argmax(axis=1)]
    cohort = np.where(gallery_labels[None,:]==leaders[:,None],-np.inf,qg)
    q_cohort = np.take_along_axis(cohort,top(cohort,k),axis=1)
    q = (qg-q_cohort.mean(axis=1,keepdims=True))/q_cohort.std(axis=1,keepdims=True).clip(.015)
    g = (qg-g_cohort.mean(axis=1))/g_cohort.std(axis=1).clip(.015)
    return .5*(q+g)

SET = 'retrieval_candidates'
top_rows = top

@dataclass
class Block:
    episode: object
    labels: np.ndarray
    scores: np.ndarray
    features: np.ndarray
    novelty: np.ndarray
    graph: sparse.csr_matrix


def zscore(x):
    return (x-x.mean(axis=1,keepdims=True))/x.std(axis=1,keepdims=True).clip(1e-6)


def graph_signals(z,group):
    n = len(z)
    ng = len(group)
    nn, vv = [], []
    for offset in range(0,n,128):
        sim = z[offset:offset+128] @ z.T
        sim[np.arange(len(sim)),offset+np.arange(len(sim))] = -2
        top = top_rows(sim,16)
        nn.append(top)
        vv.append(np.take_along_axis(sim,top,axis=1))
    nn,vv = np.vstack(nn),np.vstack(vv)
    strength=np.maximum(vv,0)**4
    graph=sparse.csr_matrix((strength.ravel(),(np.repeat(np.arange(n),16),nn.ravel())),shape=(n,n))
    graph=(graph+graph.T)*.5
    mass=np.asarray(graph.sum(1)).ravel().clip(1e-8)
    scale=sparse.diags(1/np.sqrt(mass))
    laplacian=scale@graph@scale
    y=np.zeros((n,group.max()+1),dtype='f')
    y[np.arange(ng),group]=1/np.sqrt(np.bincount(group)[group])
    current=y.copy()
    for _ in range(7):
        current=y+.77*(laplacian@current)
    class_mass=current[ng:]
    class_mass/=class_mass.sum(1,keepdims=True).clip(1e-8)
    # Correct for the different query/gallery sizes in CV and inference.
    query_frac=((nn[ng:,:10]>=ng).sum(1)+.5)/11
    enrichment=logit(query_frac)-logit((n-ng)/n)
    return np.log(class_mass.clip(1e-7)),enrichment


def class_view_stats(gallery,query,group,vg,vq):
    groups=label_groups(group)
    centroids=[]; columns=[]; tags=[]
    for label,refs in enumerate(groups):
        for view in np.unique(vg[refs]):
            centroids.append(gallery[refs[vg[refs]==view]].mean(0))
            columns.append(label)
            tags.append(view)
    sim=query@unit(np.stack(centroids)).T
    columns=np.asarray(columns)
    best=strongest(sim,columns,len(groups))
    aligned=np.where(vq[:,None]==np.asarray(tags)[None,:],sim,-1)
    return best,strongest(aligned,columns,len(groups))


def pool_peers(raw,z,view):
    sim=raw@raw.T
    np.fill_diagonal(sim,-2)
    nn=top_rows(sim,4)
    ii=np.repeat(np.arange(len(nn)),nn.shape[1]); jj=nn.ravel()
    mutual=np.any(nn[jj]==ii[:,None],axis=1)
    same_side=(view[ii]==view[jj]) | (view[ii]=="") | (view[jj]=="")
    score=np.sum(z[ii]*z[jj],axis=1)
    weights=np.where(mutual & same_side & (sim[ii,jj]>.66) & (score>.60),sim[ii,jj]**6,0)
    return sparse.csr_matrix((weights,(ii,jj)),shape=sim.shape)


def gallery_evidence(data,ep,extras,retrieval_features,geometry,tex,indices,g):
    q=ep.rows[indices]
    species=ep.species[indices[0]]
    g=g[data.species[g]==species]
    g=g[np.argsort(data.label[g],kind='stable')]
    labels,group,counts=np.unique(data.label[g],return_inverse=True,return_counts=True)
    rows=np.r_[g,q]; n=len(g)
    def fit_embedding(power):
        raw=np.sign(data.x[rows])*np.abs(data.x[rows])**.85
        pca=PCA(n_components=320,svd_solver='randomized',random_state=875,iterated_power=3)
        z=pca.fit_transform(unit(raw))
        return unit(z*(pca.explained_variance_+.0015)**(-power/2)).astype('f')
    z=fit_embedding(.58)
    subgroups=label_groups(group)
    g_z,q_z=z[:n],z[n:]
    lda=LinearDiscriminantAnalysis(solver='eigen',shrinkage=.22,
                    n_components=min(160,np.count_nonzero(counts>=2)-1))
    enough=counts[group]>=2
    lda.fit(g_z[enough],group[enough])
    l_z=unit(lda.transform(z)).astype('f')
    wg,wq=fit_wccn(g_z,subgroups,q_z)
    kg,kq=fit_kissme(g_z,subgroups,q_z,reg_s=.14,reg_d=.14)
    cg,cq=fit_cross_view_cca(g_z,subgroups,data.view[g],q_z,data.view[q],
                           max_dim=160,n_comp=72)
    raw=data.x[q]@data.x[g].T
    win=strongest(raw,group,len(labels))
    pca_sim=q_z@g_z.T
    raw_imp = data.x[g]@data.x[g].T
    norm=snorm(raw,raw_imp,group)
    basis=np.concatenate([z,.60*l_z],axis=1)
    Y=np.eye(len(labels),dtype='f')[group]
    gram=basis[:n].T@basis[:n]
    gram.flat[::len(gram)+1]+=4.
    balanced=np.linalg.solve(gram,basis[:n].T@(Y/counts[None,:]**.70))
    ridge=basis[n:]@unit(balanced.T).T
    view,same_view=class_view_stats(g_z,q_z,group,data.view[g],data.view[q])
    diff,enrichment=graph_signals(unit(np.column_stack([.75*z,.55*l_z,.35*np.vstack([kg,kq])])),group)
    f,v=tex
    fv=q_texture(f,g,q,group,len(labels))
    vl=q_texture(v,g,q,group,len(labels))
    matrices=[win,strongest(pca_sim,group,len(labels)),strongest(l_z[n:]@l_z[:n].T,group,len(labels)),
              strongest(wq@wg.T,group,len(labels)),strongest(kq@kg.T,group,len(labels)),
              strongest(cq@cg.T,group,len(labels)),view,same_view,ridge,diff,fv,vl,
              strongest(norm,group,len(labels))]
    # A compact hand-set retrieval fusion with equal input scales. An expanded
    # shortlist also admits identities backed by one specialist.
    weights=[.23,.15,.11,.08,.065,.035,.08,.025,.07,.065,.02,.025,.045]
    fusion=sum(w*zscore(mat) for w,mat in zip(weights,matrices))
    direct=top_rows(fusion,WIDTH)
    specialists=[top_rows(mat,4) for mat in (win,view,ridge,diff,fv,vl)]
    old_index=np.searchsorted(labels,ep.candidates[indices])
    assert np.array_equal(labels[old_index],ep.candidates[indices])
    out=np.empty_like(direct)
    for i in range(len(out)):
        preferred=[*direct[i,:8],*old_index[i,:5]]
        for k in range(3):
            preferred.extend(block[i,k] for block in specialists)
        preferred.extend(direct[i,8:])
        out[i] = list(dict.fromkeys(preferred))[:WIDTH]
    cand=labels[out]
    pick=np.arange(len(q))[:,None]
    old_match=cand[:,:,None]==ep.candidates[indices,None,:]
    old_pos=old_match.argmax(axis=2)
    preserved=retrieval_features[indices][pick,old_pos]
    preserved=np.where(old_match.any(2)[:,:,None],preserved,np.nan)
    extra_geometry=np.zeros((*cand.shape,4),dtype='f')
    matched=old_match.any(2)
    if geometry is not None:
        extra_geometry=np.where(matched[:,:,None],geometry[indices][pick,old_pos],0)
    ordered=np.stack([mat[pick,out] for mat in matrices],axis=2)
    relative=np.stack([mat[pick,out]-mat.max(1,keepdims=True) for mat in matrices],axis=2)
    shape=np.stack([np.log1p(counts[out]),fusion[pick,out],matched],axis=2)
    x=np.concatenate([ordered,relative,shape,extra_geometry,preserved],axis=2).astype('f')
    novelty=[ep.query_features[indices],extras['kernel_q'][indices],
              ep.sift_features[indices][:,:,[1,2,4]].max(1),enrichment]
    # Novelty inputs exclude hard-coded domain priors and query ID counters.
    for mat in [*matrices,fusion]:
        top=np.take_along_axis(mat,top_rows(mat,3),axis=1)
        novelty.append(np.column_stack([top[:,0],top[:,0]-top[:,1],top[:,0]-top[:,2]]))
    novelty=np.column_stack(novelty).astype('f')
    return cand,x,fusion[pick,out],novelty,pool_peers(data.x[q],q_z,data.view[q])


def q_texture(x,g,q,group,n):
    source=x[g].astype('f'); query=x[q].astype('f')
    center=source.mean(0)
    return strongest(unit(query-center)@unit(source-center).T,group,n)


def read_geometry(ep,key):
    path=CACHE/f'gallery_sift_{key}.npz'
    if not path.exists():return None
    with np.load(path,allow_pickle=False) as f:
        geom=f['features'][:,:,3:]
    full=(len(ep.rows),ep.candidates.shape[1],4)
    if geom.shape==full:return geom
    positions=training_positions(ep)
    assert len(geom)==len(positions)
    result=np.zeros(full,dtype='f')
    result[positions]=geom
    return result


def make_block(data,folds,key,tex):
    cache=CACHE/f'{SET}_{key}.npz'
    if cache.exists():
        ep,_=validate_inputs(CACHE,key,folds,data)
        with np.load(cache,allow_pickle=False) as f:
            assert np.array_equal(f['rows'],ep.rows)
            graph=sparse.csr_matrix((f['gv'],f['gi'],f['gp']),shape=(len(ep.rows),len(ep.rows)))
            return Block(ep,f['labels'],f['scores'],f['x'],f['novelty'],graph)
    item=retrieval_fold(data,folds,CACHE,key)
    ep,extra=item[0]
    arrays=[]
    geometry=read_geometry(ep,key)
    for sp in SPECIES:
        idx=np.flatnonzero(ep.species==sp)
        arrays.append(gallery_evidence(data,ep,extra,item[1],geometry,tex,idx,folds[key][0]))
        print(key,sp,'features ready',flush=True)
    labels,x,scores,novelty=map(np.concatenate,zip(*(block[:4] for block in arrays)))
    graph=sparse.block_diag([block[-1] for block in arrays],format='csr')
    np.savez_compressed(cache,rows=ep.rows,labels=labels,scores=scores,x=x,novelty=novelty,
                       gv=graph.data,gi=graph.indices,gp=graph.indptr)
    return Block(ep,labels,scores,x,novelty,graph)


## Rankers, novelty calibration and final blend


In [ ]:
def fit_candidates(blocks,exclude_rows=None):
    features=np.concatenate([b.features for b in blocks])
    labels=np.concatenate([b.labels for b in blocks])
    truth=np.concatenate([b.episode.labels for b in blocks])
    correct=labels==truth[:,None]
    use=correct.any(1)
    if exclude_rows is not None:
        queries=np.concatenate([b.episode.rows for b in blocks])
        use &= ~np.isin(queries,exclude_rows)
    x=features[use].reshape(-1,features.shape[2]);y=correct[use].ravel()
    light=light_model(165).fit(x,y.astype(int),group=np.full(use.sum(),WIDTH))
    hist=histogram().fit(x,y,sample_weight=np.where(y,7.,1.))
    return light,hist


def scores(models,block):
    x=block.features.reshape(-1,block.features.shape[2])
    return np.stack([light_predict(models[0],x).reshape(block.scores.shape),
                     models[1].decision_function(x).reshape(block.scores.shape)])


def novelty_inputs(block,margins):
    raw=[]
    for matrix in margins:
        best=np.take_along_axis(matrix,top(matrix,3),axis=1)
        raw.append(np.column_stack([best[:,0],best[:,0]-best[:,1],best[:,0]-best[:,-1]]))
    return np.column_stack([block.novelty,*raw])

def gallery_only_inputs(b):
    # First eleven values are legacy batch statistics. Retain metadata and
    # best gallery scores, omit query density and query-to-query affinity.
    columns = [5,6,7,8,9,29,30,31]
    # Each retrieval method adds (best gallery score, two short margins).
    # Omit diffusion and fused scores, which include query-batch evidence.
    for method in (0,1,2,3,4,5,6,7,8,10,11,12):
        columns.extend(range(33+3*method,36+3*method))
    assert b.novelty.shape[1]==75
    selected = b.novelty[:,columns]
    sift = b.novelty[:,30]
    direct = b.novelty[:,33]
    normalized = b.novelty[:,33+3*12]
    return np.column_stack([selected,sift*direct,sift*np.maximum(normalized,0)])


def fit_raw(parts,targets,profile):
    # Cache only model margins generated from supplied features. Stored labels
    # are never consumed during test scoring or calibration.
    key = hashlib.sha256(b"".join(b.episode.rows.tobytes() for b in [*parts,*targets])).hexdigest()[:20]
    file = CACHE/f"identity_ranker_{profile}_{key}.npz"
    count = len(parts)
    if file.exists():
        with np.load(file,allow_pickle=False) as stored:
            values = [stored[f"margin{i}"] for i in range(count+len(targets))]
        return values[:count],values[count:]
    held=[]
    for i, block in enumerate(parts):
        trained=fit_candidates([other for j,other in enumerate(parts) if i!=j],
                                  exclude_rows=block.episode.rows)
        held.append(scores(trained,block))
        print(profile,"finished cross-fit",i+1,flush=True)
    trained=fit_candidates(parts)
    predicted=[scores(trained,block) for block in targets]
    temp=file.with_suffix(".partial.npz")
    np.savez_compressed(temp,**{f"margin{i}":x for i,x in enumerate([*held,*predicted])})
    temp.replace(file)
    return held,predicted


def verified_graph(b,data,key,workers=4):
    file=CACHE/f"verified_query_graph_{key}.npz"
    n=len(b.episode.rows)
    if file.exists():
        with np.load(file,allow_pickle=False) as z:
            assert np.array_equal(z['rows'],b.episode.rows)
            return sparse.csr_matrix((z['w'],(z['left'],z['right'])),shape=(n,n))
    graph=b.graph.tocsr()
    graph.eliminate_zeros()
    coo=sparse.triu(graph,1).tocoo()
    rows=b.episode.rows
    pairs=list(zip(rows[coo.row],rows[coo.col]))
    good=[]
    with Verification(data,workers,CACHE) as sift:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            for offset in range(0,len(pairs),64):
                batch=pairs[offset:offset+64]
                sift.stage_images({int(index) for pair in batch for index in pair},pool)
                good.extend(pool.map(sift.compare,batch))
    mask=np.zeros(len(pairs),dtype=bool)
    weights=np.zeros(len(mask))
    if good:
        count,ratio,spread=np.asarray(good).T
        mask=(count>=10)&(ratio>=.55)&(spread>=.20)
        weights=np.minimum(count/30,1)*ratio*np.minimum(spread/.36,1)
    i=coo.row[mask];j=coo.col[mask]; w=weights[mask]
    left=np.r_[i,j];right=np.r_[j,i]; ww=np.r_[w,w]
    np.savez(file,rows=rows,w=ww,left=left,right=right)
    print(key,"verified",int(mask.sum()),"of",len(pairs),"query edges",flush=True)
    return sparse.csr_matrix((ww,(left,right)),shape=(n,n))


def geometry_pool(graph,p):
    # Trust physical matches for movement between ambiguous identities.
    # Suppress contradictory high-confidence neighbours to avoid cascading errors.
    links=graph.tocoo()
    leading=p.argmax(1)
    confidence=p.max(1)
    agree=(leading[links.row]==leading[links.col])
    both_certain=(confidence[links.row]>.80)&(confidence[links.col]>.80)
    accept=agree | ~both_certain
    weights=links.data[accept]
    matrix=sparse.csr_matrix((weights,(links.row[accept],links.col[accept])),shape=graph.shape)
    total=np.asarray(matrix.sum(1)).ravel()
    signal=(matrix@p)/total[:,None].clip(1e-12)
    alpha=.40*np.clip(total/.55,0,1)[:,None]
    return (1-alpha)*p+alpha*signal

def log_odds(model, features):
    return logit(np.clip(model.predict_proba(features)[:, 1], 1e-5, 1 - 1e-5))


def temperature(scores, classes, labels):
    match = classes == labels[:, None]
    keep = match.any(axis=1)
    s, correct = scores[keep], match[keep].argmax(axis=1)
    scale = float(np.median(np.std(s, axis=1)))
    search = np.maximum(scale, 1e-5) * np.geomspace(0.06, 1.30, 25)
    rows = np.arange(len(correct))
    nll = [-np.log(np.maximum(softmax(s / t, axis=1)[rows, correct], 1e-12)).mean()
           for t in search]
    return float(search[np.argmin(nll)])


def estimate_prior(odds, center=0.30, shrink=0.65):
    # The novelty models were fitted with a 50:50 prior within each species.
    prior = center
    for _ in range(30):
        update = float(expit(odds + logit(prior)).mean())
        if abs(update - prior) < 0.0001:
            break
        prior = float(np.clip(update, 0.08, 0.70))
    return float(np.clip(center + shrink * (prior - center), 0.12, 0.62))

def side_group(angles):
    return np.where((angles >= 5) & (angles <= 7), -1,
                    np.where((angles >= 1) & (angles <= 3), 1, 0))

@dataclass
class CameraEpisode:
    block: object
    camera: np.ndarray
    context: np.ndarray


def camera_evidence(data, gallery, query, candidates):
    """Candidate-side catalogue coverage and side-conditioned match evidence."""
    gallery = gallery[np.argsort(data.label[gallery], kind="stable")]
    labels, group, counts = np.unique(data.label[gallery], return_inverse=True, return_counts=True)
    index = np.searchsorted(labels, candidates)
    assert np.array_equal(labels[index], candidates)
    gal_side = side_group(data.angle[gallery])
    query_side = side_group(data.angle[query])
    g = data.x[gallery]
    exact_views = np.column_stack([
        np.bincount(group, weights=(data.view[gallery] == angle), minlength=len(labels))
        for angle in np.unique(data.view[query])
    ])
    exact = exact_views[index, np.searchsorted(np.unique(data.view[query]), data.view[query])[:, None]]
    features = np.zeros((*candidates.shape, 11), dtype=np.float32)
    features[:, :, 0] = np.log1p(exact)
    for side_value in (-1, 0, 1):
        key = np.flatnonzero(query_side == side_value)
        if not len(key):
            continue
        same = (gal_side == side_value) & (side_value != 0)
        opposite = (gal_side == -side_value) & (side_value != 0)
        same_n = np.bincount(group, weights=same, minlength=len(labels))
        opposite_n = np.bincount(group, weights=opposite, minlength=len(labels))
        chosen = index[key]
        features[key, :, 1:5] = np.stack([
            np.log1p(same_n[chosen]), np.log1p(opposite_n[chosen]),
            (same_n[chosen] > 0) & (opposite_n[chosen] > 0),
            (same_n[chosen] / counts[chosen]),
        ], axis=2)
        for mask, start in [(same, 5), (opposite, 8)]:
            if not mask.any():
                continue
            for batch in np.array_split(key, (len(key) + 191) // 192):
                sim = data.x[query[batch]] @ g.T
                sim[:, ~mask] = -2
                best = strongest(sim, group, len(labels))
                hits = best[np.arange(len(batch))[:, None], index[batch]]
                first_two = np.take_along_axis(best, top(best, 2), axis=1)
                rival = np.where(index[batch] == best.argmax(1)[:, None], first_two[:, 1, None],
                                 first_two[:, 0, None])
                valid = (hits > -1.5) & (rival > -1.5)
                features[batch, :, start] = np.where(hits > -1.5, hits, 0)
                features[batch, :, start + 1] = np.where(valid, hits - rival, 0)
                features[batch, :, start + 2] = valid
    return features


def augment(data, gallery, block):
    ep = block.episode
    camera = np.zeros((*block.labels.shape, 11), dtype=np.float32)
    for sp in SPECIES:
        rows = np.flatnonzero(ep.species == sp)
        camera[rows] = camera_evidence(data, gallery[data.species[gallery] == sp],
                                      ep.rows[rows], block.labels[rows])
    angle = data.angle[ep.rows]
    context = np.column_stack([ep.species == "hyena", angle < 0, angle == 0, angle == 4,
                               side_group(angle) == -1, side_group(angle) == 1])
    return CameraEpisode(block, camera, context)


def joint_features(item):
    block, camera = item.block, item.camera
    n, width = block.labels.shape
    novelty = np.column_stack([block.novelty, item.context, camera[:, :, 5:].max(axis=1)])
    feature = np.full((n, width + 1, block.features.shape[2] + camera.shape[2] +
                       novelty.shape[1] + 1), np.nan, dtype=np.float32)
    feature[:, :-1, :block.features.shape[2]] = block.features
    feature[:, :-1, block.features.shape[2]:-novelty.shape[1]-1] = camera
    feature[:, :, -novelty.shape[1]-1:-1] = novelty[:, None, :]
    feature[:, :, -1] = (np.arange(width + 1) == width).astype(np.float32)
    return feature


def fit_joint(parts, predict, excluded=None):
    x = np.concatenate([joint_features(item) for item in parts])
    labels = np.concatenate([item.block.labels for item in parts])
    truth = np.concatenate([item.block.episode.labels for item in parts])
    rows = np.concatenate([item.block.episode.rows for item in parts])
    species = np.concatenate([item.block.episode.species for item in parts])
    matches = labels == truth[:, None]
    novel = truth == NEW
    use = matches.any(1) | novel
    if excluded is not None:
        use &= ~np.isin(rows, excluded)
    x, y = x[use], np.column_stack([matches, novel])[use]
    width = x.shape[1]
    flat, target = x.reshape(-1, x.shape[2]), y.ravel().astype(int)
    weights = np.ones(len(x))
    # Equal known/novel weights within species reduce sensitivity to the split's
    # chosen mix. The supervised novelty stage estimates transfer separately.
    for sp in SPECIES:
        part = species[use] == sp
        for flag in (True, False):
            group = part & (novel[use] == flag)
            weights[group] = .5 * part.sum() / max(group.sum(), 1)
    cat = CatBoostRanker(iterations=145, depth=5, learning_rate=.064, l2_leaf_reg=13,
                         random_seed=2026, thread_count=2, loss_function="QuerySoftMax",
                         verbose=False, allow_writing_files=False)
    cat.fit(flat, target, group_id=np.repeat(np.arange(len(y)), width),
            group_weight=np.repeat(weights, width))
    xgb = XGBRanker(n_estimators=135, max_depth=4, learning_rate=.06, reg_lambda=15,
                    min_child_weight=14, subsample=.89, colsample_bytree=.82,
                    objective="rank:ndcg", lambdarank_num_pair_per_sample=7,
                    tree_method="hist", n_jobs=2, random_state=2026)
    xgb.fit(flat, target, group=np.full(len(y), width), sample_weight=weights)
    estimates = []
    for item in predict:
        x = joint_features(item)
        z = x.reshape(-1, x.shape[2])
        estimates.append(np.stack([model.predict(z).reshape(x.shape[:2]) for model in (cat, xgb)]))
    return estimates


def joint_crossfits(parts, targets, key):
    fingerprint = hashlib.sha256(b"".join(item.block.episode.rows.tobytes()
                                          for item in [*parts, *targets])).hexdigest()[:16]
    path = CACHE / f"joint_ranker_{key}_{fingerprint}.npz"
    if not path.exists():
        values = []
        for index, item in enumerate(parts):
            others = [part for j, part in enumerate(parts) if j != index]
            values.extend(fit_joint(others, [item], item.block.episode.rows))
            print(key, "joint ranker cross-fit", index + 1, flush=True)
        values.extend(fit_joint(parts, targets))
        temp = path.with_suffix(".partial.npz")
        np.savez_compressed(temp, *values)
        temp.replace(path)
    with np.load(path, allow_pickle=False) as stored:
        values = [stored[f"arr_{i}"] for i in range(len(parts) + len(targets))]
    return values[:len(parts)], values[len(parts):]


def uncalibrated_features(item, base, joint, stable):
    b = item.block
    # Describe the two leading identities. Only the observed candidates' side
    # coverage is used. Gallery coverage for the true identity is audit-only.
    blend = .5 * base[0] / base[0].std(axis=1, keepdims=True).clip(.01) + \
            .5 * base[1] / base[1].std(axis=1, keepdims=True).clip(.01)
    leaders = top(blend, 2)
    position = np.arange(len(leaders))[:, None]
    camera = item.camera[position, leaders]
    key_signals = b.features[position, leaders][:, :, [0, 1, 2, 6, 7, 8, 10, 11, 12, 26, 29, 30, 31, 32]]
    lead = np.column_stack([camera[:, 0], key_signals[:, 0], camera[:, 1, [5, 6, 8, 9]],
                            key_signals[:, 0] - key_signals[:, 1], item.context])
    inputs = gallery_only_inputs(b) if stable else novelty_inputs(b, base)
    if stable:
        return np.column_stack([inputs, lead])
    extra = [score[:, -1] - score[:, :-1].max(1) for score in joint]
    return np.column_stack([inputs, lead, *extra])


def fit_calibrators(parts, base_margins, joint_margins):
    y = np.concatenate([item.block.episode.novel for item in parts])
    species = np.concatenate([item.block.episode.species for item in parts])
    full = np.concatenate([uncalibrated_features(*items, False)
                           for items in zip(parts, base_margins, joint_margins)])
    stable = np.concatenate([uncalibrated_features(*items, True)
                             for items in zip(parts, base_margins, joint_margins)])
    # Compress sample weights for queries appearing in multiple inner episodes.
    rows = np.concatenate([item.block.episode.rows for item in parts])
    _, group, count = np.unique(rows, return_inverse=True, return_counts=True)
    weight = 1 / count[group]
    for sp in SPECIES:
        for label in (True, False):
            use = (species == sp) & (y == label)
            weight[use] *= .5 * (species == sp).sum() / weight[use].sum()
    logistic = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                             LogisticRegression(C=.035, max_iter=450))
    logistic.fit(stable, y, logisticregression__sample_weight=weight)
    trees = make_pipeline(SimpleImputer(strategy="median"),
                         ExtraTreesClassifier(n_estimators=210, max_features=.68,
                                              min_samples_leaf=22, max_depth=11, n_jobs=3,
                                              random_state=26120))
    trees.fit(full, y, extratreesclassifier__sample_weight=weight)
    return logistic, trees


def joint_temperatures(raw, parts):
    truth = np.concatenate([item.block.episode.labels for item in parts])
    labels = np.concatenate([item.block.labels for item in parts])
    labels = np.column_stack([labels, np.full(len(labels), NEW)])
    return [temperature(np.concatenate([score[j] for score in raw]), labels, truth)
            for j in range(len(raw[0]))]


def dense(labels, probabilities, vocab):
    result = np.zeros((len(labels), len(vocab) + 1))
    result[np.arange(len(labels))[:, None], np.searchsorted(vocab, labels)] = probabilities[:, :-1]
    result[:, -1] = probabilities[:, -1]
    return result


In [ ]:
def fit_graph_calibration(parts, held):
    truth = np.concatenate([block.episode.labels for block in parts])
    species = np.concatenate([block.episode.species for block in parts])
    features = np.concatenate([novelty_inputs(block, scores) for block, scores in zip(parts, held)])
    unseen = truth == NEW
    weights = np.ones(len(unseen))
    for sp in SPECIES:
        group = species == sp
        for label in (False, True):
            use = group & (unseen == label)
            weights[use] = 0.5 * group.sum() / max(use.sum(), 1)
    logistic = make_pipeline(StandardScaler(), LogisticRegression(C=.05,max_iter=500))
    logistic.fit(features, unseen, logisticregression__sample_weight=weights)
    hist = HistGradientBoostingClassifier(max_iter=105, learning_rate=.085, max_leaf_nodes=15,
                                         min_samples_leaf=55, l2_regularization=6.0,
                                         early_stopping=False, random_state=392)
    hist.fit(features, unseen, sample_weight=weights)
    labels = np.concatenate([block.labels for block in parts])
    temps = [temperature(np.concatenate([m[k] for m in held]), labels, truth) for k in (0,1)]
    return logistic, hist, temps


def portfolio_posterior(data, item, base, joint, graph_fit, side_models, joint_temps, vocab):
    block = item.block
    ep = block.episode
    logistic, hist, temperatures = graph_fit
    known = .76*softmax(base[0]/temperatures[0],axis=1) + .24*softmax(base[1]/temperatures[1],axis=1)
    graph = verified_graph(block, data, "test", workers=WORKERS)
    features = novelty_inputs(block,base)
    odds = .50*log_odds(logistic,features) + .50*log_odds(hist,features)
    graph_p = geometry_pool(graph, dense_posterior((block.labels,known,odds),ep.species,vocab))
    side_logistic, side_forest = side_models
    forest = log_odds(side_forest, uncalibrated_features(item,base,joint,False))
    linear = log_odds(side_logistic, uncalibrated_features(item,base,joint,True))
    side_p = geometry_pool(graph, dense_posterior(
        (block.labels,known,.55*forest+.45*linear),ep.species,vocab))
    cat,xgb = [dense(block.labels, softmax(margin/t,axis=1),vocab)
               for margin,t in zip(joint,joint_temps)]
    joint_p = .50*cat + .50*xgb
    hyena = (ep.species=="hyena")[:,None]
    weights = [np.where(hyena, h, l) for h,l in ((.50,.15),(.40,.70),(.10,.15))]
    return sum(weight*p for weight,p in zip(weights,(graph_p,side_p,joint_p)))


## Generate the scored submission


In [ ]:
EXPECTED_SHA256 = "3bfada239fddab77aaf4daf8be409c770ee37a45fba3a4592c76ee10abe7b8be"


def generate_submission(data_root, output):
    output.mkdir(parents=True,exist_ok=True)
    CACHE.mkdir(parents=True,exist_ok=True)
    data = Data(data_root)
    training = [f"full_inner_{i}" for i in range(3)]
    keys = [*training,"test"]
    folds = dict(zip(training,calibration_splits(data,data.train_rows)))
    folds["test"] = (data.train_rows,data.test_rows)
    store = SiftStore(data,WORKERS)
    try:
        prepare(data,folds,store,CACHE,keys)
    finally:
        store.close()
    needed = [CACHE/"texture_codes.npz", *(CACHE/f"gallery_sift_{key}.npz" for key in keys)]
    if all(path.exists() for path in needed):
        tex = texture_codes(data,folds,keys,None)
    else:
        with Verification(data,WORKERS,CACHE) as verifier:
            for key in keys:
                prepare_geometry(data,folds,key,verifier)
            tex = texture_codes(data,folds,keys,verifier)
    parts = [augment(data,folds[key][0],make_block(data,folds,key,tex)) for key in keys]
    train,target = parts[:-1],parts[-1]
    training_blocks = [part.block for part in train]
    held,base = fit_raw(training_blocks,[target.block],"test")
    joint_held,joint = joint_crossfits(train,[target],"test")
    graph_fit = fit_graph_calibration(training_blocks,held)
    side_fit = fit_calibrators(train,held,joint_held)
    temps = joint_temperatures(joint_held,train)
    vocabulary = np.union1d(possible_labels(validate_inputs(CACHE,"test",folds,data)),target.block.labels)
    posterior = portfolio_posterior(data,target,base[0],joint[0],graph_fit,side_fit,temps,vocabulary)
    np.testing.assert_allclose(posterior.sum(1),1)
    prediction = ranked(posterior,vocabulary)
    assert prediction.shape == (2791,5)
    path = output/"submssion_species_specific.csv"
    write_submission(data,target.block.episode,prediction,path)
    signature = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path} | {len(prediction):,} rows")
    print(f"SHA-256 {signature}")
    print(f"Exact match to the scored CSV {signature == EXPECTED_SHA256}")
    return path


submission = generate_submission(DATA_ROOT, OUTPUT)
